In [ ]:
# # 九曜：天墟 LF01 — 分情緒參考聲音試做（2026-10-05）
# 每角色用原提示詞＋情緒指示（低聲緊張／喊叫）、原種子設計情緒參考句，再用 Base 照唸 2 句，看同一角色在不同情緒下音色是否一致。


In [ ]:
import subprocess
print(subprocess.run("pip install -q -U qwen-tts soundfile 2>&1 | tail -3; nvidia-smi --query-gpu=name --format=csv", shell=True, capture_output=True, text=True).stdout)

In [ ]:
import json, os, base64, subprocess
DESIGN = json.loads(r'''[["机遙_低聲_s33", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。男主角机遥，二十岁的成年男性青年，男声，中低音，绝对不是少年音、童声或女声。声音有厚度、沉稳，语速正常偏快，说话干脆利落、不拖音，平时温和，关键时刻坚定。这一句压低声音、气声多、紧张急促，像怕被近在眼前的敌人听见。", "别出声……它就在雾里，离我们不到十步。", 33], ["机遙_低聲_s11", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。男主角机遥，二十岁的成年男性青年，男声，中低音，绝对不是少年音、童声或女声。声音有厚度、沉稳，语速正常偏快，说话干脆利落、不拖音，平时温和，关键时刻坚定。这一句压低声音、气声多、紧张急促，像怕被近在眼前的敌人听见。", "别出声……它就在雾里，离我们不到十步。", 11], ["机遙_喊叫_s33", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。男主角机遥，二十岁的成年男性青年，男声，中低音，绝对不是少年音、童声或女声。声音有厚度、沉稳，语速正常偏快，说话干脆利落、不拖音，平时温和，关键时刻坚定。这一句是战斗中用尽全力的大喊，声音洪亮、撕裂、情绪激烈。", "所有人退后——快！别让它靠近阵眼！", 33], ["机遙_喊叫_s11", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。男主角机遥，二十岁的成年男性青年，男声，中低音，绝对不是少年音、童声或女声。声音有厚度、沉稳，语速正常偏快，说话干脆利落、不拖音，平时温和，关键时刻坚定。这一句是战斗中用尽全力的大喊，声音洪亮、撕裂、情绪激烈。", "所有人退后——快！别让它靠近阵眼！", 11], ["江祈璟_低聲_s11", "热血玄幻国漫（3D修仙动画）的配音，标准普通话，专业配音演员的表演腔，情绪饱满、咬字清楚。二十岁枪修青年，沉稳冷峻的男声，音调中低、声音干净有力。骄傲自持、带一点审视人的冷淡，说话简洁果断、不拖泥带水，关键时刻语气沉下来、有担当。这一句压低声音、气声多、紧张急促，像怕被近在眼前的敌人听见。", "别出声……它就在雾里，离我们不到十步。", 11], ["江祈璟_低聲_s22", "热血玄幻国漫（3D修仙动画）的配音，标准普通话，专业配音演员的表演腔，情绪饱满、咬字清楚。二十岁枪修青年，沉稳冷峻的男声，音调中低、声音干净有力。骄傲自持、带一点审视人的冷淡，说话简洁果断、不拖泥带水，关键时刻语气沉下来、有担当。这一句压低声音、气声多、紧张急促，像怕被近在眼前的敌人听见。", "别出声……它就在雾里，离我们不到十步。", 22], ["江祈璟_喊叫_s11", "热血玄幻国漫（3D修仙动画）的配音，标准普通话，专业配音演员的表演腔，情绪饱满、咬字清楚。二十岁枪修青年，沉稳冷峻的男声，音调中低、声音干净有力。骄傲自持、带一点审视人的冷淡，说话简洁果断、不拖泥带水，关键时刻语气沉下来、有担当。这一句是战斗中用尽全力的大喊，声音洪亮、撕裂、情绪激烈。", "所有人退后——快！别让它靠近阵眼！", 11], ["江祈璟_喊叫_s22", "热血玄幻国漫（3D修仙动画）的配音，标准普通话，专业配音演员的表演腔，情绪饱满、咬字清楚。二十岁枪修青年，沉稳冷峻的男声，音调中低、声音干净有力。骄傲自持、带一点审视人的冷淡，说话简洁果断、不拖泥带水，关键时刻语气沉下来、有担当。这一句是战斗中用尽全力的大喊，声音洪亮、撕裂、情绪激烈。", "所有人退后——快！别让它靠近阵眼！", 22], ["聞人澈_低聲_s22", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。二十岁法修青年，温润清朗的男声，音调中低，带一点书卷气。语速正常偏快、干脆利落、不拖音；身处战斗之中，语气紧张急促、坚定，像在护着身后的人。这一句压低声音、气声多、紧张急促，像怕被近在眼前的敌人听见。", "别出声……它就在雾里，离我们不到十步。", 22], ["聞人澈_低聲_s11", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。二十岁法修青年，温润清朗的男声，音调中低，带一点书卷气。语速正常偏快、干脆利落、不拖音；身处战斗之中，语气紧张急促、坚定，像在护着身后的人。这一句压低声音、气声多、紧张急促，像怕被近在眼前的敌人听见。", "别出声……它就在雾里，离我们不到十步。", 11], ["聞人澈_喊叫_s22", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。二十岁法修青年，温润清朗的男声，音调中低，带一点书卷气。语速正常偏快、干脆利落、不拖音；身处战斗之中，语气紧张急促、坚定，像在护着身后的人。这一句是战斗中用尽全力的大喊，声音洪亮、撕裂、情绪激烈。", "所有人退后——快！别让它靠近阵眼！", 22], ["聞人澈_喊叫_s11", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。二十岁法修青年，温润清朗的男声，音调中低，带一点书卷气。语速正常偏快、干脆利落、不拖音；身处战斗之中，语气紧张急促、坚定，像在护着身后的人。这一句是战斗中用尽全力的大喊，声音洪亮、撕裂、情绪激烈。", "所有人退后——快！别让它靠近阵眼！", 11], ["郁岑燁_低聲_s22", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。二十岁天才剑修，高冷孤傲的青年男声，音色清冷干净、像冰一样，音调中等偏高。语速偏慢、语气淡漠疏离，带著居高临下的轻蔑，懒得多说一个字，句尾轻轻下压，一句话就让人觉得被看不起。这一句压低声音、气声多、紧张急促，像怕被近在眼前的敌人听见。", "别出声……它就在雾里，离我们不到十步。", 22], ["郁岑燁_低聲_s33", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。二十岁天才剑修，高冷孤傲的青年男声，音色清冷干净、像冰一样，音调中等偏高。语速偏慢、语气淡漠疏离，带著居高临下的轻蔑，懒得多说一个字，句尾轻轻下压，一句话就让人觉得被看不起。这一句压低声音、气声多、紧张急促，像怕被近在眼前的敌人听见。", "别出声……它就在雾里，离我们不到十步。", 33], ["郁岑燁_喊叫_s22", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。二十岁天才剑修，高冷孤傲的青年男声，音色清冷干净、像冰一样，音调中等偏高。语速偏慢、语气淡漠疏离，带著居高临下的轻蔑，懒得多说一个字，句尾轻轻下压，一句话就让人觉得被看不起。这一句是战斗中用尽全力的大喊，声音洪亮、撕裂、情绪激烈。", "所有人退后——快！别让它靠近阵眼！", 22], ["郁岑燁_喊叫_s33", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。二十岁天才剑修，高冷孤傲的青年男声，音色清冷干净、像冰一样，音调中等偏高。语速偏慢、语气淡漠疏离，带著居高临下的轻蔑，懒得多说一个字，句尾轻轻下压，一句话就让人觉得被看不起。这一句是战斗中用尽全力的大喊，声音洪亮、撕裂、情绪激烈。", "所有人退后——快！别让它靠近阵眼！", 33]]''')
TEST = json.loads(r'''[["机遙_低聲_s33", "别出声……它就在雾里，离我们不到十步。", ["别跟它说话。", "……它在看我们。"]], ["机遙_低聲_s11", "别出声……它就在雾里，离我们不到十步。", ["别跟它说话。", "……它在看我们。"]], ["机遙_喊叫_s33", "所有人退后——快！别让它靠近阵眼！", ["右边断了！", "全部退开——！"]], ["机遙_喊叫_s11", "所有人退后——快！别让它靠近阵眼！", ["右边断了！", "全部退开——！"]], ["江祈璟_低聲_s11", "别出声……它就在雾里，离我们不到十步。", ["别跟它说话。", "……它在看我们。"]], ["江祈璟_低聲_s22", "别出声……它就在雾里，离我们不到十步。", ["别跟它说话。", "……它在看我们。"]], ["江祈璟_喊叫_s11", "所有人退后——快！别让它靠近阵眼！", ["右边断了！", "全部退开——！"]], ["江祈璟_喊叫_s22", "所有人退后——快！别让它靠近阵眼！", ["右边断了！", "全部退开——！"]], ["聞人澈_低聲_s22", "别出声……它就在雾里，离我们不到十步。", ["别跟它说话。", "……它在看我们。"]], ["聞人澈_低聲_s11", "别出声……它就在雾里，离我们不到十步。", ["别跟它说话。", "……它在看我们。"]], ["聞人澈_喊叫_s22", "所有人退后——快！别让它靠近阵眼！", ["右边断了！", "全部退开——！"]], ["聞人澈_喊叫_s11", "所有人退后——快！别让它靠近阵眼！", ["右边断了！", "全部退开——！"]], ["郁岑燁_低聲_s22", "别出声……它就在雾里，离我们不到十步。", ["别跟它说话。", "……它在看我们。"]], ["郁岑燁_低聲_s33", "别出声……它就在雾里，离我们不到十步。", ["别跟它说话。", "……它在看我们。"]], ["郁岑燁_喊叫_s22", "所有人退后——快！别让它靠近阵眼！", ["右边断了！", "全部退开——！"]], ["郁岑燁_喊叫_s33", "所有人退后——快！别让它靠近阵眼！", ["右边断了！", "全部退开——！"]]]''')
OUT='/kaggle/working/out'; os.makedirs(OUT, exist_ok=True); os.makedirs('/tmp/ref', exist_ok=True)


In [ ]:
import torch, soundfile as sf, gc
from qwen_tts import Qwen3TTSModel
design = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=torch.float32, attn_implementation="sdpa")
for name, prompt, text, s in DESIGN:
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    wavs, sr = design.generate_voice_design(text=text, language="Chinese", instruct=prompt)
    sf.write(f"/tmp/ref/{name}.wav", wavs[0][: int(sr * 12)], sr); sf.write(f"{OUT}/REF_{name}.wav", wavs[0][: int(sr * 12)], sr)
    print(name, f"{len(wavs[0])/sr:.2f}s")
del design; gc.collect(); torch.cuda.empty_cache()


In [ ]:
clone = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0", dtype=torch.float32, attn_implementation="sdpa")
for name, reftext, lines in TEST:
    w, sr0 = sf.read(f"/tmp/ref/{name}.wav")
    pr = clone.create_voice_clone_prompt(ref_audio=(w, sr0), ref_text=reftext)
    for i, text in enumerate(lines, 1):
        try:
            wavs, sr = clone.generate_voice_clone(text=text, language="Chinese", voice_clone_prompt=pr, max_new_tokens=12 * 10)
        except TypeError:
            wavs, sr = clone.generate_voice_clone(text=text, language="Chinese", voice_clone_prompt=pr)
        sf.write(f"{OUT}/{name}_{i:02d}.wav", wavs[0][: int(sr * 10)], sr); print(name, i, text)


In [ ]:
import glob
for w in sorted(glob.glob(f"{OUT}/*.wav")):
    m = w[:-4] + ".mp3"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", w, "-ac", "1", "-b:a", "96k", m], check=True)
    print(f"@@MP3 {os.path.basename(m)} {base64.b64encode(open(m, 'rb').read()).decode()}")
